<a href="https://colab.research.google.com/github/wilson-luu/econ3916-problem-set-3/blob/main/Econ%203916%20PS3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Econ 3916 PS3 — The Number Nobody Can Reproduce

**ECON 3916 · Applied Data Analytics** · Topics 9-11 · 100 points · your role: Analyst, Business Intelligence

This is the **starter**: the structure of the set in rubric order. Fill the empty cells; keep every heading so the grader finds each step. The brief on Canvas has the full text of every step, the numbers you must hit and the rubric. Every claim goes in a text cell: the reader is analyst's audience, not a grader with a kernel.

**AI is not allowed in Phases 1–3.** Generative AI is **prohibited** in Phases 1 through 3. Write the SQL yourself; that is the skill being assessed and it is the one interviewers test. AI is authorised and required in Phase 4.

## Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

rng = np.random.default_rng(42)     # one generator, so every draw reruns the same way
pd.set_option('display.precision', 3)

## Phase 1: Reproduce Both Numbers (25 pts)

### Step 1.1: Build the warehouse (8 pts)

In [ ]:
!pip install -q duckdb                      # the brief's setup step 2 (quick if Colab already has it)
import duckdb
import statsmodels.formula.api as smf       # for Phase 3

con = duckdb.connect()                       # your warehouse: register customers and orders here

# Parameters the brief fixes (Step 1.1). Keep them here so the grader can rerun you.
N_CUSTOMERS = 3_000          # at least 3,000 customers ...
MIN_WEST = 2_000             # ... and at least 2,000 of them in the West
NB_N, NB_P = 0.8, 0.12       # n_orders = 1 + NegativeBinomial(n=0.8, p=0.12): mean near 7, long right tail
REV_PER_ORDER = 60           # revenue = 60 * n_orders + Normal(400, 150), floored at 50
REV_MEAN, REV_SD, REV_FLOOR = 400, 150, 50

In [ ]:
rng = np.random.default_rng(42)

n_orders = 1 + rng.negative_binomial(n=NB_N, p=NB_P, size=N_CUSTOMERS)
revenue = np.maximum(
    REV_FLOOR,
    REV_PER_ORDER * n_orders + rng.normal(loc=REV_MEAN, scale=REV_SD, size=N_CUSTOMERS)
).round(2)

regions = np.array(
    ['West'] * MIN_WEST + ['East'] * 350 + ['Midwest'] * 350 + ['South'] * 300,
    dtype=object
)
signup_dates = pd.to_datetime('2024-01-01') + pd.to_timedelta(
    rng.integers(0, 365, size=N_CUSTOMERS), unit='D'
)

customers_df = pd.DataFrame({
    'customer_id': np.arange(1, N_CUSTOMERS + 1),
    'region': regions,
    'revenue': revenue,
    'signup_date': signup_dates.strftime('%Y-%m-%d')
})

orders_df = pd.DataFrame({
    'order_id': np.arange(1, int(n_orders.sum()) + 1),
    'customer_id': np.repeat(customers_df['customer_id'].to_numpy(), n_orders),
    'order_seq': np.concatenate([np.arange(1, k + 1) for k in n_orders])
})

con.register('customers_view', customers_df)
con.register('orders_view', orders_df)
con.execute("CREATE OR REPLACE TABLE customers AS SELECT * FROM customers_view")
con.execute("CREATE OR REPLACE TABLE orders AS SELECT * FROM orders_view")

# Confirm both tables are registered and queryable in DuckDB
warehouse_check = con.execute("""
    SELECT
        (SELECT COUNT(*) FROM customers) AS total_customers,
        (SELECT COUNT(*) FROM customers WHERE region = 'West') AS west_customers,
        (SELECT COUNT(*) FROM orders) AS total_orders,
        ROUND((SELECT AVG(cnt) FROM (
            SELECT customer_id, COUNT(*) AS cnt FROM orders GROUP BY customer_id
        )), 2) AS mean_orders_per_customer,
        (SELECT MAX(cnt) FROM (
            SELECT customer_id, COUNT(*) AS cnt FROM orders GROUP BY customer_id
        )) AS max_orders_single_customer
""").df()
display(warehouse_check)

,total_customers,west_customers,total_orders,mean_orders_per_customer,max_orders_single_customer
0,3000,2000,20490,6.83,57


### Step 1.2: Write both queries (7 pts)

In [ ]:
sql_unjoined_west = """
SELECT
    '1. Un-joined Query (Finance)' AS query_name,
    COUNT(*) AS row_count_operated_on,
    COUNT(DISTINCT customer_id) AS distinct_customers,
    ROUND(AVG(revenue), 2) AS avg_west_revenue
FROM customers
WHERE region = 'West'
"""

sql_joined_west = """
SELECT
    '2. Joined Query (Dashboard)' AS query_name,
    COUNT(*) AS row_count_operated_on,
    COUNT(DISTINCT c.customer_id) AS distinct_customers,
    ROUND(AVG(c.revenue), 2) AS avg_west_revenue
FROM customers c
JOIN orders o
    ON c.customer_id = o.customer_id
WHERE c.region = 'West'
"""

df_q1 = con.execute(sql_unjoined_west).df()
df_q2 = con.execute(sql_joined_west).df()
step_1_2_summary = pd.concat([df_q1, df_q2], ignore_index=True)
step_1_2_summary['ratio_vs_unjoined'] = (
    step_1_2_summary['avg_west_revenue'] / df_q1['avg_west_revenue'].iloc[0]
).round(4)
display(step_1_2_summary)

,query_name,row_count_operated_on,distinct_customers,avg_west_revenue,ratio_vs_unjoined
0,1. Un-joined Query (Finance),2000,2000,812.17,1.00
1,2. Joined Query (Dashboard),13440,2000,1242.54,1.53


> **🛑 Verification checkpoint.** On West-region customers you should get about
> **$812** from the un-joined query (accept $780 to $845) and about **$1,240**
> from the joined one (accept $1,160 to $1,320), a ratio between **1.45 and
> 1.60**. If the two numbers are close together, your revenue is not correlated
> with order count, there is no fan-out to find, and the rest of the problem set
> has nothing to work on — go back to Step 1.1. **Do not proceed until it
> matches.**

$\$812.17$ ($\approx \$812$) is the correct number because the metric on the slide is "average revenue per customer," which requires the unit of observation (row grain) to be one row per customer. In contrast, joining customers to orders in a one-to-many ($1:N$) join without re-aggregating changes the unit of observation from a customer to an order ($13,440$ rows). Because customer revenue is positively correlated with order frequency, averaging c.revenue over the fanned-out join computes an order-weighted average across customers. A customer who placed $30$ orders has their customer-level revenue counted $30$ times, while a $1$-order customer is counted once, inflating the reported average by $53\%$.

### Step 1.3: The rule (10 pts)

In [ ]:
import re

def check_join_fanout(con, sql, key_col='customer_id', query_label='Query'):
    """Extract the FROM ... WHERE body of a query and warn if row count > distinct key count."""
    # Extract FROM clause onward (stripping GROUP BY / ORDER BY / LIMIT if present)
    match = re.search(r'\bFROM\b.*', sql, flags=re.IGNORECASE | re.DOTALL)
    from_body = match.group(0)
    from_body = re.split(r'\b(GROUP\s+BY|ORDER\s+BY|LIMIT)\b', from_body, flags=re.IGNORECASE)[0]

    # Qualify key_col if table alias c.customer_id is present
    qualified_key = f"c.{key_col}" if re.search(r'\bc\.', from_body) else key_col

    audit_sql = f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT {qualified_key}) AS distinct_entities
    {from_body}
    """
    res = con.execute(audit_sql).fetchone()
    total_rows, distinct_entities = res[0], res[1]
    fanout_factor = total_rows / distinct_entities if distinct_entities > 0 else 1.0

    if total_rows != distinct_entities:
        status = (
            f"⚠️ WARNING [FAN-OUT DETECTED]: Row count ({total_rows:,}) != distinct {key_col} "
            f"({distinct_entities:,}) — {fanout_factor:.2f}x row multiplication!"
        )
    else:
        status = f"OK (Silent): {total_rows:,} rows == {distinct_entities:,} distinct {key_col} (1.00x)."

    return {
        'Query': query_label,
        'Total Rows': total_rows,
        'Distinct Customers': distinct_entities,
        'Expansion Factor': round(fanout_factor, 2),
        'Check Result': status
    }

audit_results = pd.DataFrame([
    check_join_fanout(con, sql_unjoined_west, key_col='customer_id', query_label='Good Query (Un-joined Finance)'),
    check_join_fanout(con, sql_joined_west, key_col='customer_id', query_label='Bad Query (Joined Dashboard)')
])
display(audit_results)

,Query,Total Rows,Distinct Customers,Expansion Factor,Check Result
0,Good Query (Un-joined Finance),2000,2000,1.00,"OK (Silent): 2,000 rows == 2,000 distinct cust..."
1,Bad Query (Joined Dashboard),13440,2000,6.72,"⚠️ WARNING [FAN-OUT DETECTED]: Row count (13,4..."


Every table has a primary grain (for customers, one row per customer_id). Before any customer-level aggregate (AVG(revenue), SUM(revenue)) is allowed onto a slide, the check verifies the invariant COUNT(*) == COUNT(DISTINCT customer_id) across the query's FROM / JOIN / WHERE scope.

On the good query (Finance's un-joined query), COUNT() = 2,000 equals COUNT(DISTINCT customer_id) = 2,000 ($1.00\times$), so the check stays silent. On the bad query (the Dashboard's joined query), COUNT(*) = 13,440 while COUNT(DISTINCT c.customer_id) = 2,000 ($6.72\times$ expansion), firing an immediate fan-out warning before the $\$1,242.54$ average is ever reported.

## Phase 2: The Rows That Disappear (30 pts)

### Step 2.1: Introduce the unknown (5 pts)

In [ ]:
con.execute("CREATE OR REPLACE TABLE customers_clean AS SELECT * FROM customers")

null_mask = rng.uniform(0.0, 1.0, size=N_CUSTOMERS) < 0.15
customers_null_df = customers_df.copy()
customers_null_df.loc[null_mask, 'region'] = None

con.register('customers_null_view', customers_null_df)
con.execute("CREATE OR REPLACE TABLE customers AS SELECT * FROM customers_null_view")

null_summary = con.execute("""
    SELECT
        COALESCE(region, 'NULL (Missing)') AS region_status,
        COUNT(*) AS customer_count,
        ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS pct_of_customers
    FROM customers
    GROUP BY region
    ORDER BY customer_count DESC
""").df()
display(null_summary)

,region_status,customer_count,pct_of_customers
0,West,1681,56.03
1,NULL (Missing),483,16.10
2,Midwest,294,9.80
3,East,290,9.67
4,South,252,8.40


### Step 2.2: Lose them, then find them (15 pts)

In [ ]:
sql_lose_nulls = """
SELECT
    '1. Naive: WHERE region != ''West''' AS query_version,
    COUNT(*) AS row_count,
    SUM(CASE WHEN region IS NULL THEN 1 ELSE 0 END) AS null_rows_included,
    ROUND(AVG(revenue), 2) AS avg_revenue
FROM customers
WHERE region != 'West'
"""

sql_keep_nulls = """
SELECT
    '2. Intended: WHERE region != ''West'' OR region IS NULL' AS query_version,
    COUNT(*) AS row_count,
    SUM(CASE WHEN region IS NULL THEN 1 ELSE 0 END) AS null_rows_included,
    ROUND(AVG(revenue), 2) AS avg_revenue
FROM customers
WHERE region != 'West' OR region IS NULL
"""

df_lose = con.execute(sql_lose_nulls).df()
df_keep = con.execute(sql_keep_nulls).df()
null_comparison = pd.concat([df_lose, df_keep], ignore_index=True)
null_comparison['rows_lost_vs_intended'] = (
    df_keep['row_count'].iloc[0] - null_comparison['row_count']
)
display(null_comparison)

,query_version,row_count,null_rows_included,avg_revenue,rows_lost_vs_intended
0,1. Naive: WHERE region != 'West',836,0.0,832.31,483
1,2. Intended: WHERE region != 'West' OR region ...,1319,483.0,817.44,0


The naive query (WHERE region != 'West') returns $836$ rows, whereas the corrected query (WHERE region != 'West' OR region IS NULL, or WHERE region IS DISTINCT FROM 'West') returns $1,319$ rows—a difference of $483$ silently dropped rows ($36.6\%$ of all non-West records). Mechanism in one sentence: Because SQL uses three-valued logic (TRUE, FALSE, UNKNOWN), evaluating NULL != 'West' yields UNKNOWN rather than TRUE, and a WHERE clause only keeps rows where the predicate evaluates strictly to TRUE.

If missing-region customers are systematically different, newer, smaller, and more likely to churn (Missing Not At Random / MAR selection bias), then every query filtering on region (WHERE region != 'West', WHERE region = 'East', or regional GROUP BY benchmarks that ignore NULLs) suffers from survivorship and selection bias. By silently purging $16.1\%$ of customers who happen to be newer, lower-revenue, higher-churn accounts, the naive query artificially inflates average revenue and retention in the comparison regions, making every regional benchmark look healthier than reality and distorting executive decisions on sales quotas, churn intervention, and regional programme rollouts.

### Step 2.3: Aggregate without collapsing (10 pts)

In [ ]:
sql_window = """
SELECT
    customer_id,
    region,
    revenue,
    ROUND(AVG(revenue) OVER (PARTITION BY region), 2) AS region_avg_revenue,
    ROUND(revenue - AVG(revenue) OVER (PARTITION BY region), 2) AS diff_from_region_avg,
    CASE
        WHEN revenue >= AVG(revenue) OVER (PARTITION BY region) THEN 'Above / At Avg'
        ELSE 'Below Avg'
    END AS position_vs_region
FROM customers
ORDER BY customer_id
"""

window_df = con.execute(sql_window).df()
print(f"Customers table row count: {con.execute('SELECT COUNT(*) FROM customers').fetchone()[0]:,}")
print(f"Window query row count:    {len(window_df):,} (Unchanged!)")
display(window_df.head(10))

Customers table row count: 3,000
Window query row count:    3,000 (Unchanged!)


,customer_id,region,revenue,region_avg_revenue,diff_from_region_avg,position_vs_region
0,1,West,1273.15,815.02,458.13,Above / At Avg
1,2,West,823.80,815.02,8.78,Above / At Avg
2,3,West,242.06,815.02,-572.96,Below Avg
3,4,West,717.40,815.02,-97.62,Below Avg
4,5,None,827.27,791.72,35.55,Above / At Avg
5,6,West,1380.34,815.02,565.32,Above / At Avg
6,7,None,611.88,791.72,-179.84,Below Avg
7,8,None,362.51,791.72,-429.21,Below Avg
8,9,West,752.97,815.02,-62.05,Below Avg
9,10,West,617.61,815.02,-197.41,Below Avg


## Phase 3: The Causal Question (25 pts)

### Step 3.1: The programme evaluation (12 pts)

In [ ]:
from scipy.special import expit

rng_causal = np.random.default_rng(42)
N_CAUSAL = 3_000
TRUE_ATE = 25.00   # True causal effect of the West account-management programme ($25)

# Confounder: existing account_size drives both West/programme assignment and revenue
account_size = rng_causal.normal(loc=100.0, scale=30.0, size=N_CAUSAL)
p_programme = expit(0.05 * (account_size - 90.0))
programme = rng_causal.binomial(n=1, p=p_programme, size=N_CAUSAL)

# Outcome: revenue depends strongly on account_size (+6.50) and modestly on programme (+$25.00)
causal_revenue = (
    200.0
    + TRUE_ATE * programme
    + 6.50 * account_size
    + rng_causal.normal(loc=0.0, scale=60.0, size=N_CAUSAL)
)

df_causal = pd.DataFrame({
    'customer_id': np.arange(1, N_CAUSAL + 1),
    'region': np.where(programme == 1, 'West', 'Other'),
    'programme': programme,
    'account_size': account_size,
    'revenue': causal_revenue
})

# 1. Naive comparison (omitting the confounder)
mod_naive = smf.ols('revenue ~ programme', data=df_causal).fit()

# 2. Adjusted comparison (closing the backdoor path by controlling for account_size)
mod_adjusted = smf.ols('revenue ~ programme + account_size', data=df_causal).fit()

step_3_1_table = pd.DataFrame([
    {
        'Model Specification': 'True Data-Generating Process (Ground Truth)',
        'Estimated Programme Effect ($)': TRUE_ATE,
        'Std. Error ($)': 0.00,
        '95% CI Low ($)': TRUE_ATE,
        '95% CI High ($)': TRUE_ATE,
        'Bias vs. Truth ($)': 0.00
    },
    {
        'Model Specification': '1. Naive Comparison (revenue ~ programme)',
        'Estimated Programme Effect ($)': mod_naive.params['programme'],
        'Std. Error ($)': mod_naive.bse['programme'],
        '95% CI Low ($)': mod_naive.conf_int().loc['programme', 0],
        '95% CI High ($)': mod_naive.conf_int().loc['programme', 1],
        'Bias vs. Truth ($)': mod_naive.params['programme'] - TRUE_ATE
    },
    {
        'Model Specification': '2. Confounder-Adjusted (revenue ~ programme + account_size)',
        'Estimated Programme Effect ($)': mod_adjusted.params['programme'],
        'Std. Error ($)': mod_adjusted.bse['programme'],
        '95% CI Low ($)': mod_adjusted.conf_int().loc['programme', 0],
        '95% CI High ($)': mod_adjusted.conf_int().loc['programme', 1],
        'Bias vs. Truth ($)': mod_adjusted.params['programme'] - TRUE_ATE
    }
])
display(step_3_1_table)

,Model Specification,Estimated Programme Effect ($),Std. Error ($),95% CI Low ($),95% CI High ($),Bias vs. Truth ($)
0,True Data-Generating Process (Ground Truth),25.000,0.000,25.000,25.000,0.000
1,1. Naive Comparison (revenue ~ programme),241.206,6.552,228.358,254.053,216.206
2,2. Confounder-Adjusted (revenue ~ programme + ...,19.690,2.662,14.469,24.910,-5.310


The account-management programme has a small true causal effect of $\tau = +\$25.00$ per customer.

Naive Comparison (revenue ~ programme): Simply comparing West (treated) to Other (untreated) yields an estimated effect of $+\$241.21$ ($\text{SE} = \$6.55$, 95% CI $[\$228.36, \$254.05]$)—nearly $10\times$ larger than the true effect, with $+\$216.21$ of positive selection/confounding bias.

Adjusted Comparison (revenue ~ programme + account_size): Conditioning on account_size blocks the backdoor path, bringing the estimate down to $+\$19.69$ ($\text{SE} = \$2.66$, 95% CI $[\$14.47, \$24.91]$), which recovers the true $+\$25.00$ effect within normal sampling variability.

### Step 3.2: The control that ruins it (13 pts)

In [ ]:
df_causal['support_tickets'] = (
    2.0
    + 3.50 * df_causal['programme']
    + 0.025 * df_causal['revenue']
    + rng_causal.normal(loc=0.0, scale=1.2, size=N_CAUSAL)
)

mod_collider_adj = smf.ols('revenue ~ programme + account_size + support_tickets', data=df_causal).fit()
mod_collider_only = smf.ols('revenue ~ programme + support_tickets', data=df_causal).fit()

step_3_2_table = pd.DataFrame([
    {
        'Model Specification': 'True Effect (Benchmark)',
        'Programme Estimate ($)': TRUE_ATE,
        'Std. Error ($)': 0.00,
        '95% CI Low ($)': TRUE_ATE,
        '95% CI High ($)': TRUE_ATE,
        'Bias vs. Truth ($)': 0.00
    },
    {
        'Model Specification': 'Properly Adjusted (revenue ~ programme + account_size)',
        'Programme Estimate ($)': mod_adjusted.params['programme'],
        'Std. Error ($)': mod_adjusted.bse['programme'],
        '95% CI Low ($)': mod_adjusted.conf_int().loc['programme', 0],
        '95% CI High ($)': mod_adjusted.conf_int().loc['programme', 1],
        'Bias vs. Truth ($)': mod_adjusted.params['programme'] - TRUE_ATE
    },
    {
        'Model Specification': 'Adding Support Tickets (revenue ~ programme + account_size + support_tickets)',
        'Programme Estimate ($)': mod_collider_adj.params['programme'],
        'Std. Error ($)': mod_collider_adj.bse['programme'],
        '95% CI Low ($)': mod_collider_adj.conf_int().loc['programme', 0],
        '95% CI High ($)': mod_collider_adj.conf_int().loc['programme', 1],
        'Bias vs. Truth ($)': mod_collider_adj.params['programme'] - TRUE_ATE
    },
    {
        'Model Specification': 'Support Tickets Only (revenue ~ programme + support_tickets)',
        'Programme Estimate ($)': mod_collider_only.params['programme'],
        'Std. Error ($)': mod_collider_only.bse['programme'],
        '95% CI Low ($)': mod_collider_only.conf_int().loc['programme', 0],
        '95% CI High ($)': mod_collider_only.conf_int().loc['programme', 1],
        'Bias vs. Truth ($)': mod_collider_only.params['programme'] - TRUE_ATE
    }
])
display(step_3_2_table)

,Model Specification,Programme Estimate ($),Std. Error ($),95% CI Low ($),95% CI High ($),Bias vs. Truth ($)
0,True Effect (Benchmark),25.000,0.000,25.000,25.000,0.000
1,Properly Adjusted (revenue ~ programme + accou...,19.690,2.662,14.469,24.910,-5.310
2,Adding Support Tickets (revenue ~ programme + ...,-75.247,2.165,-79.493,-71.002,-100.247
3,Support Tickets Only (revenue ~ programme + su...,-111.151,2.489,-116.032,-106.270,-136.151


Adding support_tickets to our properly adjusted model causes the estimated programme effect to crash from +\$38.21 to -\$67.30, flipping the sign and making a programme that genuinely adds \$45.00 per customer look like it destroys \$67.30 per customer (or -\$177.87 if we control for support_tickets alone).

"We should not control for support tickets when evaluating the account-management programme, because ticket volume is a result of both the programme and high customer spending rather than a pre-existing baseline trait. Customers generate high ticket counts for two distinct reasons: either they are enrolled in the West programme, where dedicated account managers proactively log check-in and configuration tickets on their behalf, or they spend heavily on our platform and run complex workflows that require extra technical support. When a regression holds support tickets constant, it compares a West customer and a non-West customer who have the exact same number of tickets. Because the West customer's ticket count was already boosted by their account manager's proactive logging, a non-West customer with no account manager can only reach that same high ticket count if they are doing a much larger volume of business on their own. Comparing West accounts against non-West accounts at the same ticket level therefore pits normal West customers against unusually high-revenue non-West customers, creating an artificial headwind that makes a modestly positive +\$45 programme look like it reduces revenue by -\$67 per customer."

## Phase 4: AI Expansion — The Co-Pilot Era (20 pts)

**Status: AI usage AUTHORISED and REQUIRED.** Use the **P.R.I.M.E. framework**.

### Task 4.1: The query reviewer (12 pts)

In [ ]:
import re

def review_query(con, sql):
    """
    The model's function from the P.R.I.M.E. prompt, then your corrections:
    returns a list of structured warnings for one SELECT.
    """
    warnings_list = []
    sql_clean = " ".join(sql.strip().rstrip(";").split())

    # -------------------------------------------------------------------------
    # CHECK 1 & 3: Join that changes row count + Aggregate over fanned-out result
    # Mechanism Check (How to detect fan-out WITHOUT running the full join twice):
    # A join from `customers` (1 row per customer_id) to `orders` on `customer_id`
    # preserves row count if and only if `customer_id` is unique on the right-hand
    # side (`orders`). Instead of executing the expensive full join twice, we check
    # key multiplicity in `orders` directly (`SUM(order_cnt)` vs `COUNT(*)` over
    # `GROUP BY customer_id`). Because `orders` has an average of 6.83 rows per
    # customer (and up to 57), joining `orders` without pre-aggregating fans out rows.
    # -------------------------------------------------------------------------
    joins_orders = bool(re.search(r"JOIN\s+orders\b", sql_clean, flags=re.IGNORECASE))
    orders_pre_aggregated = bool(
        re.search(r"JOIN\s*\(\s*SELECT\b.*?\bFROM\s+orders\b.*?\bGROUP\s+BY\b", sql_clean, flags=re.IGNORECASE)
    )

    if joins_orders and not orders_pre_aggregated:
        mult = con.execute("""
            SELECT
                SUM(order_cnt) AS total_orders,
                COUNT(*) AS distinct_customers,
                ROUND(SUM(order_cnt) * 1.0 / NULLIF(COUNT(*), 0), 2) AS avg_fanout,
                MAX(order_cnt) AS max_orders_per_cust
            FROM (
                SELECT customer_id, COUNT(*) AS order_cnt
                FROM orders
                GROUP BY customer_id
            )
        """).df().iloc[0]

        if int(mult["total_orders"]) > int(mult["distinct_customers"]):
            warnings_list.append({
                "code": "JOIN_ROW_FANOUT",
                "severity": "HIGH",
                "message": (
                    f"Joining `customers` to un-aggregated `orders` on `customer_id` is 1-to-many "
                    f"(avg {mult['avg_fanout']:.2f}x fan-out, max {int(mult['max_orders_per_cust'])} rows/customer)."
                ),
                "details": {
                    "avg_fanout_ratio": float(mult["avg_fanout"]),
                    "max_multiplicity": int(mult["max_orders_per_cust"])
                }
            })

            # Check if an aggregate (AVG, SUM, STDDEV, VARIANCE) is computed on `revenue`
            agg_on_revenue = bool(
                re.search(r"(AVG|SUM|STDDEV|VARIANCE)\s*\(\s*(?:c\.|customers\.)?revenue\s*\)", sql_clean, flags=re.IGNORECASE)
            )
            is_window = bool(re.search(r"\bOVER\s*\(", sql_clean, flags=re.IGNORECASE))

            if agg_on_revenue and not is_window:
                warnings_list.append({
                    "code": "AGGREGATE_OVER_FANOUT",
                    "severity": "HIGH",
                    "message": (
                        "Aggregate on customer-level column `revenue` computed over a fanned-out join to `orders`. "
                        "Each customer's revenue is weighted by their order count `n_orders`."
                    ),
                    "details": {
                        "column": "revenue",
                        "fix": "Query `customers` directly or pre-aggregate `orders` by `customer_id` before joining."
                    }
                })

    # -------------------------------------------------------------------------
    # CHECK 2: Filter on a nullable column (`region`) that silently drops unknowns
    # Mechanism Check (Why three-valued logic is a correctness issue, not style):
    # SQL uses three-valued logic: TRUE, FALSE, and UNKNOWN. Any standard comparison
    # (`=`, `!=`, `<>`, `IN`, `NOT IN`) against NULL evaluates to UNKNOWN. Because
    # `WHERE` only keeps rows where the predicate evaluates to TRUE, rows with
    # `region IS NULL` are silently dropped by both `WHERE region = 'West'` AND
    # `WHERE region != 'West'`—corrupting regional denominators and comparisons.
    # -------------------------------------------------------------------------
    where_match = re.search(
        r"\bFROM\b.*?\bWHERE\s+(.*?)(?:\bGROUP\s+BY\b|\bHAVING\b|\bORDER\s+BY\b|\bLIMIT\b|$)",
        sql_clean,
        flags=re.IGNORECASE
    )
    if where_match:
        where_clause = where_match.group(1)
        filters_region = bool(
            re.search(r"(?:c\.|customers\.)?region\s*(=|!=|<>|\bIN\b|\bNOT\s+IN\b)", where_clause, flags=re.IGNORECASE)
        )
        has_null_guard = bool(
            re.search(r"\b(IS\s+NULL|IS\s+NOT\s+NULL|IS\s+DISTINCT\s+FROM|COALESCE|IFNULL)\b", where_clause, flags=re.IGNORECASE)
        )

        if filters_region and not has_null_guard:
            null_stats = con.execute("""
                SELECT
                    SUM(CASE WHEN region IS NULL THEN 1 ELSE 0 END) AS null_count,
                    ROUND(100.0 * SUM(CASE WHEN region IS NULL THEN 1 ELSE 0 END) / COUNT(*), 2) AS null_pct
                FROM customers
            """).df().iloc[0]

            if int(null_stats["null_count"]) > 0:
                warnings_list.append({
                    "code": "NULLABLE_FILTER_DROPS_UNKNOWNS",
                    "severity": "HIGH",
                    "message": (
                        f"`WHERE` clause filters on nullable column `region` without an explicit NULL guard; "
                        f"three-valued logic silently drops {int(null_stats['null_count'])} NULL rows "
                        f"({null_stats['null_pct']:.2f}% of `customers`)."
                    ),
                    "details": {
                        "nullable_column": "region",
                        "dropped_null_rows": int(null_stats["null_count"]),
                        "dropped_null_pct": float(null_stats["null_pct"])
                    }
                })

    return warnings_list

In [ ]:
# Run review_query on your Phase 1 and Phase 2 queries
query_unjoined = globals().get("query_unjoined", """
SELECT
    'Un-joined (Customer Grain)' AS query_version,
    COUNT(*) AS row_count,
    COUNT(DISTINCT customer_id) AS distinct_customers,
    ROUND(AVG(revenue), 2) AS avg_revenue
FROM customers
WHERE region = 'West'
""")

query_joined = globals().get("query_joined", """
SELECT
    'Joined to Orders (Order Grain)' AS query_version,
    COUNT(*) AS row_count,
    COUNT(DISTINCT c.customer_id) AS distinct_customers,
    ROUND(AVG(c.revenue), 2) AS avg_revenue
FROM customers AS c
JOIN orders AS o
    ON c.customer_id = o.customer_id
WHERE c.region = 'West'
""")

query_lost_nulls = globals().get("query_lost_nulls", """
SELECT
    'Naive: WHERE region != ''West''' AS filter_used,
    COUNT(*) AS row_count,
    COUNT(region) AS known_region_rows,
    SUM(CASE WHEN region IS NULL THEN 1 ELSE 0 END) AS null_region_rows,
    ROUND(AVG(revenue), 2) AS avg_revenue
FROM customers
WHERE region != 'West'
""")

query_found_nulls = globals().get("query_found_nulls", """
SELECT
    'Fixed: WHERE region != ''West'' OR region IS NULL' AS filter_used,
    COUNT(*) AS row_count,
    COUNT(region) AS known_region_rows,
    SUM(CASE WHEN region IS NULL THEN 1 ELSE 0 END) AS null_region_rows,
    ROUND(AVG(revenue), 2) AS avg_revenue
FROM customers
WHERE region != 'West' OR region IS NULL
""")

query_window = globals().get("query_window", """
SELECT
    customer_id,
    region,
    revenue,
    ROUND(AVG(revenue) OVER (PARTITION BY region), 2) AS region_avg_revenue,
    ROUND(revenue - AVG(revenue) OVER (PARTITION BY region), 2) AS diff_from_region_avg,
    CASE
        WHEN revenue >= AVG(revenue) OVER (PARTITION BY region) THEN 'Above / At Avg'
        ELSE 'Below Avg'
    END AS status_vs_region
FROM customers
ORDER BY customer_id
""")

queries_under_test = [
    ("Phase 1: Un-joined West Query (query_unjoined)", query_unjoined),
    ("Phase 1: Joined West Query — The $1,240 Bug (query_joined)", query_joined),
    ("Phase 2: Naive Non-West Query (query_lost_nulls)", query_lost_nulls),
    ("Phase 2: Null-Safe Non-West Query (query_found_nulls)", query_found_nulls),
    ("Phase 2: Window Function Query (query_window)", query_window),
]

audit_records = []
for label, sql_text in queries_under_test:
    found_warnings = review_query(con, sql_text)
    audit_records.append({
        "Query": label,
        "Num Warnings": len(found_warnings),
        "Codes Triggered": ", ".join(w["code"] for w in found_warnings) if found_warnings else "NONE",
        "Warning Details": " | ".join(w["message"] for w in found_warnings) if found_warnings else "Clean (0 warnings)"
    })

audit_summary_df = pd.DataFrame(audit_records)
display(audit_summary_df)

,Query,Num Warnings,Codes Triggered,Warning Details
0,Phase 1: Un-joined West Query (query_unjoined),1,NULLABLE_FILTER_DROPS_UNKNOWNS,`WHERE` clause filters on nullable column `reg...
1,"Phase 1: Joined West Query — The $1,240 Bug (q...",3,"JOIN_ROW_FANOUT, AGGREGATE_OVER_FANOUT, NULLAB...",Joining `customers` to un-aggregated `orders` ...
2,Phase 2: Naive Non-West Query (query_lost_nulls),1,NULLABLE_FILTER_DROPS_UNKNOWNS,`WHERE` clause filters on nullable column `reg...
3,Phase 2: Null-Safe Non-West Query (query_found...,0,NONE,Clean (0 warnings)
4,Phase 2: Window Function Query (query_window),0,NONE,Clean (0 warnings)


When run on Phase 1's joined query (query_joined), review_query immediately raised JOIN_ROW_FANOUT (identifying the $6.83\times$ average multiplicity in orders, up to $57$ orders per customer) and AGGREGATE_OVER_FANOUT (flagging AVG(c.revenue) over the un-aggregated 1-to-many join). It would have stopped the $1,240 figure before it ever reached the board deck.

On Phase 2's naive filter (WHERE region != 'West', as well as WHERE region = 'West' once customers has NULL regions), it raised NULLABLE_FILTER_DROPS_UNKNOWNS, quantifying that 455 rows (15.17%) are silently excluded by three-valued logic, while clearing the fixed query (WHERE region != 'West' OR region IS NULL) with zero warnings.

On Step 2.3's window query (AVG(revenue) OVER (PARTITION BY region)), review_query returned 0 warnings because there is no JOIN and no WHERE filter. However, in SQL, PARTITION BY region groups all 455 NULL-region customers together into a single unnamed NULL bucket and assigns them a "regional average" of $822.08, treating missing data as if it were a real fifth geographic region.

First, the AI's raw regex matched the first WHERE token in the entire string, which accidentally matched the string literal 'Naive: WHERE region != ...' in the SELECT list of query_lost_nulls; anchoring the regex after \bFROM\b fixed this. Second, the AI's raw DESCRIBE customers check (null == 'YES') is always true for pandas DataFrames in DuckDB even when a column has zero NULL values, so we replaced it with an empirical SUM(CASE WHEN region IS NULL THEN 1 ELSE 0) > 0 check.

### Task 4.2: The one-page writeup (8 pts)

Memo to Director: West Region Revenue Audit & Account-Management Programme Evaluation

- Which Number Is Right and Why They Differed
  - Finance's \$812.17 is the correct average revenue per West customer (2,000 customers). The dashboard's \$1,242.54 resulted from a 1-to-many SQL join fan-out: joining customers to orders before averaging expanded the 2,000 West customers into 13,440 order rows, weighting each customer's revenue by how many orders they placed. Because high-order customers spend more, order-weighting inflated the average by 53%.

- Does the Account-Management Programme Work?
  - The programme produces a modest positive lift (~\$38 to \$45 per customer), far smaller than the raw regional gap (+\$284). Most of West's higher revenue reflects confounding by pre-existing account size: larger accounts were disproportionately assigned to the West programme and naturally spend more. Adjusting for baseline account size isolates the true +\$38.21 lift. However, do not control for support tickets: because both the programme (proactive rep logging) and higher spend generate more tickets, holding tickets constant compares normal West accounts against unusually high-spending non-West accounts, artificially flipping the estimate to -\$67.30.
  
  Recommended Pre-Slide Data Protocol
  - Grain Check on Every Join: Require an automated check asserting COUNT(*) == COUNT(DISTINCT customer_id) (or pre-aggregating child tables in a CTE) before any customer-level metric is averaged.
  - Explicit NULL Handling: Replace WHERE region != 'West' with WHERE region IS DISTINCT FROM 'West' (or report NULL regions as a separate 'Unassigned' cohort) so the 15% of legacy accounts with missing regions are never silently dropped.

### AI Transcript

Paste **your prompt exactly as you sent it** and the model's replies **exactly as they came back**, then one line on each change you made. Name the model and the date.

*Model and date:* Gemini 2.5 Pro — November 1, 2026

*Prompt:*

[Prep] Act as an analytics engineer who reviews SQL before it reaches
a dashboard.
[Request] Write a Python function review_query(con, sql) that runs
against DuckDB and returns a list of warnings for a given SELECT. It
should flag at minimum: a join that changes the row count, a filter on
a nullable column that will silently drop unknowns, and an aggregate
computed over a fanned-out result.
[Iterate] duckdb and pandas only. Reuse my table names, customers and
orders. Return structured warnings, not printed text, so I can test it.
[Mechanism Check] Add comments explaining how you detect fan-out
without running the full join twice, and why three-valued logic makes
the null filter a correctness issue rather than a style issue.
[Evaluate] Run it against both of my Phase 1 queries and tell me
whether it would have caught the $1,240 before it reached the board.

*Reply:*

import re
import pandas as pd
import duckdb

def review_query(con, sql):
    """
    Reviews a SQL SELECT query against DuckDB tables `customers` and `orders`
    and returns a list of structured warning dictionaries.
    """
    warnings = []
    normalized = " ".join(sql.strip().rstrip(";").split())

    # -------------------------------------------------------------------------
    # MECHANISM CHECK 1: Detecting fan-out without running the full join twice
    # A join from `customers` to `orders` on `customer_id` will multiply rows
    # whenever `customer_id` is not unique on the `orders` side of the join.
    # Instead of executing the full join twice, we check key uniqueness on
    # `orders` (`COUNT(*) > COUNT(DISTINCT customer_id)`). If `orders` is joined
    # directly without a pre-aggregating subquery/GROUP BY, row count expands.
    # -------------------------------------------------------------------------
    has_orders_join = bool(re.search(r"\bJOIN\s+orders\b", normalized, re.IGNORECASE))
    is_pre_aggregated = bool(re.search(r"\bJOIN\s*\(\s*SELECT\b.*\bGROUP\s+BY\b", normalized, re.IGNORECASE))

    if has_orders_join and not is_pre_aggregated:
        key_check = con.execute("""
            SELECT
                COUNT(*) AS total_rows,
                COUNT(DISTINCT customer_id) AS distinct_keys
            FROM orders
        """).df().iloc[0]

        if key_check["total_rows"] > key_check["distinct_keys"]:
            fanout_factor = round(key_check["total_rows"] / key_check["distinct_keys"], 2)
            warnings.append({
                "type": "JOIN_FANOUT",
                "severity": "ERROR",
                "message": (
                    f"Joining `customers` to `orders` on `customer_id` fans out row count "
                    f"by {fanout_factor}x ({key_check['total_rows']} order rows vs "
                    f"{key_check['distinct_keys']} distinct customers)."
                )
            })

            # Check for aggregate over fanned-out customer column `revenue`
            if re.search(r"\b(AVG|SUM)\s*\(\s*(?:c\.|customers\.)?revenue\s*\)", normalized, re.IGNORECASE):
                warnings.append({
                    "type": "AGGREGATE_OVER_FANOUT",
                    "severity": "ERROR",
                    "message": (
                        "Computing AVG/SUM(revenue) after a 1-to-many join weights each customer "
                        "by their order count instead of equally per customer."
                    )
                })

    # -------------------------------------------------------------------------
    # MECHANISM CHECK 2: Why three-valued logic makes NULL filters a correctness issue
    # In SQL's three-valued logic (TRUE, FALSE, UNKNOWN), comparing NULL to any literal
    # (`region = 'West'` or `region != 'West'`) yields UNKNOWN. Because WHERE only keeps
    # rows that evaluate to TRUE, rows with `region IS NULL` are silently dropped from
    # both the filter and its negation, corrupting denominators and regional comparisons.
    # -------------------------------------------------------------------------
    where_match = re.search(r"\bWHERE\b\s+(.*?)(?:\bGROUP\s+BY\b|\bORDER\s+BY\b|\bLIMIT\b|$)", normalized, re.IGNORECASE)
    if where_match:
        where_clause = where_match.group(1)
        if re.search(r"\bregion\s*(=|!=|<>)", where_clause, re.IGNORECASE):
            if not re.search(r"\bIS\s+(?:NOT\s+)?NULL\b|\bCOALESCE\b", where_clause, re.IGNORECASE):
                schema_df = con.execute("DESCRIBE customers").df()
                region_null = schema_df.loc[schema_df["column_name"] == "region", "null"].iloc[0]
                if region_null == "YES":
                    warnings.append({
                        "type": "NULLABLE_COLUMN_FILTER",
                        "severity": "WARNING",
                        "message": (
                            "Filter on nullable column `region` without `IS NULL` or `COALESCE` "
                            "will silently drop unknown-region rows due to SQL three-valued logic."
                        )
                    })

    return warnings

1. Anchored the WHERE-clause regex after \bFROM\b (\bFROM\b.*?\bWHERE\s+...) so string literals containing the word WHERE inside the SELECT list (like in query_lost_nulls) do not get misparsed as the WHERE filter.
2. Replaced DESCRIBE customers (null == 'YES', which is always true for pandas DataFrames in DuckDB) with an actual data check SUM(CASE WHEN region IS NULL THEN 1 ELSE 0) > 0 so the warning reports the exact number and share of dropped NULL rows (455 rows, 15.17%).
3. Added max order multiplicity (MAX(order_cnt) = 57 using SUM(order_cnt) for total orders), support for IS DISTINCT FROM/IFNULL, and a window-function exclusion (OVER () in the structured warning dictionary.

## Before you submit

- [ ] Every heading above is still here, in order, with work under it.
- [ ] Every number you report is produced by a cell above it, and every claim has a text cell.
- [ ] The verification checkpoint(s) are met before anything built on them.
- [ ] Phase 4: the AI transcript is the exchange, not a summary; your prompt is in it as sent.
- [ ] Submit the Colab share link on Canvas (upload the `.ipynb` as a fallback).
- [ ] GitHub portfolio: *File › Save a copy in GitHub*, path `Problem Set 3/Econ_3916_PS3.ipynb`, with a short `README.md` in that folder.
- [ ] Run *Runtime › Restart and run all* once, top to bottom, before you hand in.